# Model Training — MEPS 2022 Healthcare Expenditure Prediction

Trains and evaluates 4 models predicting **log1p(total annual healthcare expenditure)**.

| Model | Input | Notes |
|---|---|---|
| Linear Regression | Scaled | Baseline linear model |
| SVR (RBF kernel) | Scaled | Non-linear kernel, robust to outliers |
| Random Forest | Unscaled | Ensemble of 300 trees |
| XGBoost | Unscaled | Gradient-boosted trees |

Models are saved to `../models/` for use in the web app.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print('Libraries loaded.')

Libraries loaded.


In [2]:
# Scaled versions for Linear Regression and SVR
X_train_sc = pd.read_csv('X_train_scaled.csv')
X_test_sc  = pd.read_csv('X_test_scaled.csv')

# Unscaled versions for Random Forest and XGBoost
X_train    = pd.read_csv('../artifacts/X_train.csv')
X_test     = pd.read_csv('../artifacts/X_test.csv')

y_train = pd.read_csv('../artifacts/y_train.csv').squeeze()
y_test  = pd.read_csv('../artifacts/y_test.csv').squeeze()

feature_cols = list(X_train.columns)

print(f'Train : {X_train.shape[0]:,} rows x {X_train.shape[1]} features')
print(f'Test  : {X_test.shape[0]:,} rows')
print(f'\nTarget stats:')
print(f'  log1p train mean = {y_train.mean():.3f}  std = {y_train.std():.3f}')
print(f'  Original expenditure train mean = ${np.expm1(y_train).mean():,.0f}')

Train : 17,944 rows x 55 features
Test  : 4,487 rows

Target stats:
  log1p train mean = 6.563  std = 3.215
  Original expenditure train mean = $7,627


In [3]:
def evaluate(name, y_true, y_pred):
    """Return dict of metrics on both log and original dollar scale."""
    rmse_log = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_log  = mean_absolute_error(y_true, y_pred)
    r2       = r2_score(y_true, y_pred)
    
    # Convert back to dollar scale
    y_true_dollar = np.expm1(y_true)
    y_pred_dollar = np.expm1(y_pred)
    rmse_dollar   = np.sqrt(mean_squared_error(y_true_dollar, y_pred_dollar))
    mae_dollar    = mean_absolute_error(y_true_dollar, y_pred_dollar)
    
    return {
        'model': name,
        'r2': round(r2, 4),
        'rmse_log': round(rmse_log, 4),
        'mae_log': round(mae_log, 4),
        'rmse_dollar': round(rmse_dollar, 2),
        'mae_dollar': round(mae_dollar, 2),
    }

results = []

## 1. Linear Regression (Baseline)

In [4]:
print('Training Linear Regression...')
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_lr = lr.predict(X_test_sc)
res_lr = evaluate('Linear Regression', y_test, y_pred_lr)
results.append(res_lr)

print(f"  R²          : {res_lr['r2']:.4f}")
print(f"  RMSE (log)  : {res_lr['rmse_log']:.4f}")
print(f"  MAE  (log)  : {res_lr['mae_log']:.4f}")
print(f"  RMSE ($)    : ${res_lr['rmse_dollar']:,.0f}")
print(f"  MAE  ($)    : ${res_lr['mae_dollar']:,.0f}")

Training Linear Regression...
  R²          : 0.4049
  RMSE (log)  : 2.4783
  MAE  (log)  : 1.8677
  RMSE ($)    : $20,136
  MAE  ($)    : $6,980


## 2. Support Vector Regression (RBF Kernel)

Uses RBF kernel with C=10, epsilon=0.1. Trained on scaled data.
Takes a few minutes on 18k rows.

In [5]:
print('Training SVR (this may take 2-5 minutes)...')
svr = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
svr.fit(X_train_sc, y_train)

y_pred_svr = svr.predict(X_test_sc)
res_svr = evaluate('SVR', y_test, y_pred_svr)
results.append(res_svr)

print(f"  R²          : {res_svr['r2']:.4f}")
print(f"  RMSE (log)  : {res_svr['rmse_log']:.4f}")
print(f"  MAE  (log)  : {res_svr['mae_log']:.4f}")
print(f"  RMSE ($)    : ${res_svr['rmse_dollar']:,.0f}")
print(f"  MAE  ($)    : ${res_svr['mae_dollar']:,.0f}")

Training SVR (this may take 2-5 minutes)...
  R²          : 0.3794
  RMSE (log)  : 2.5308
  MAE  (log)  : 1.8127
  RMSE ($)    : $19,053
  MAE  ($)    : $6,756


## 3. Random Forest (300 Trees)

In [6]:
print('Training Random Forest...')
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
res_rf = evaluate('Random Forest', y_test, y_pred_rf)
results.append(res_rf)

print(f"  R²          : {res_rf['r2']:.4f}")
print(f"  RMSE (log)  : {res_rf['rmse_log']:.4f}")
print(f"  MAE  (log)  : {res_rf['mae_log']:.4f}")
print(f"  RMSE ($)    : ${res_rf['rmse_dollar']:,.0f}")
print(f"  MAE  ($)    : ${res_rf['mae_dollar']:,.0f}")

Training Random Forest...
  R²          : 0.4191
  RMSE (log)  : 2.4485
  MAE  (log)  : 1.8110
  RMSE ($)    : $18,763
  MAE  ($)    : $6,286


## 4. XGBoost

In [7]:
print('Training XGBoost...')
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)

y_pred_xgb = xgb.predict(X_test)
res_xgb = evaluate('XGBoost', y_test, y_pred_xgb)
results.append(res_xgb)

print(f"  R²          : {res_xgb['r2']:.4f}")
print(f"  RMSE (log)  : {res_xgb['rmse_log']:.4f}")
print(f"  MAE  (log)  : {res_xgb['mae_log']:.4f}")
print(f"  RMSE ($)    : ${res_xgb['rmse_dollar']:,.0f}")
print(f"  MAE  ($)    : ${res_xgb['mae_dollar']:,.0f}")

Training XGBoost...
  R²          : 0.4398
  RMSE (log)  : 2.4044
  MAE  (log)  : 1.7702
  RMSE ($)    : $18,460
  MAE  ($)    : $6,220


## 5. Results Comparison

In [8]:
results_df = pd.DataFrame(results).set_index('model')
print('=' * 70)
print('MODEL COMPARISON'.center(70))
print('=' * 70)
print(results_df[['r2', 'rmse_log', 'mae_log', 'rmse_dollar', 'mae_dollar']].to_string())
print('=' * 70)

best_model_name = results_df['r2'].idxmax()
print(f'\nBest model by R²: {best_model_name}  (R² = {results_df.loc[best_model_name, "r2"]:.4f})')

                           MODEL COMPARISON                           
                       r2  rmse_log  mae_log  rmse_dollar  mae_dollar
model                                                                
Linear Regression  0.4049    2.4783   1.8677     20136.07     6980.33
SVR                0.3794    2.5308   1.8127     19053.16     6756.39
Random Forest      0.4191    2.4485   1.8110     18763.17     6286.25
XGBoost            0.4398    2.4044   1.7702     18459.65     6220.39

Best model by R²: XGBoost  (R² = 0.4398)


In [9]:
# Feature importance for tree models
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

def plot_importance(ax, importances, title, top_n=20):
    idx = np.argsort(importances)[-top_n:]
    ax.barh(np.array(feature_cols)[idx], importances[idx], color='steelblue')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.tick_params(axis='y', labelsize=9)
    ax.invert_yaxis()

plot_importance(axes[0], rf.feature_importances_, 'Random Forest — Top 20 Features')
plot_importance(axes[1], xgb.feature_importances_, 'XGBoost — Top 20 Features')

plt.tight_layout()
plt.savefig('../models/figures/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: models/figures/feature_importance.png')

Saved: models/figures/feature_importance.png


## 6. Save Models

In [10]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(lr,  '../models/baseline/linear_regression.pkl')
joblib.dump(svr, '../models/baseline/svr.pkl')
joblib.dump(rf,  '../models/random_forest.pkl')
joblib.dump(xgb, '../models/baseline/xgboost.pkl')

# Save results summary
results_df.reset_index().to_json('../models/meta/model_results.json', orient='records', indent=2)

# Save which model is best and what type each needs (scaled vs unscaled)
model_meta = {
    'best_model': best_model_name,
    'models': {
        'Linear Regression': {'file': 'baseline/linear_regression.pkl', 'needs_scaling': True},
        'SVR':               {'file': 'baseline/svr.pkl',               'needs_scaling': True},
        'Random Forest':     {'file': 'random_forest.pkl',     'needs_scaling': False},
        'XGBoost':           {'file': 'baseline/xgboost.pkl',           'needs_scaling': False},
    },
    'scaler_file': '../artifacts/scaler.pkl',
    'feature_cols': feature_cols,
}
with open('../models/meta/model_meta.json', 'w') as f:
    json.dump(model_meta, f, indent=2)

print('Models saved to ../models/:')
for name, meta in model_meta['models'].items():
    r2_val = results_df.loc[name, 'r2']
    print(f"  {meta['file']:<30} R²={r2_val:.4f}  scaling={'yes' if meta['needs_scaling'] else 'no'}")
print(f"\nBest model: {best_model_name}")

Models saved to ../models/:
  baseline/linear_regression.pkl          R²=0.4049  scaling=yes
  baseline/svr.pkl                        R²=0.3794  scaling=yes
  random_forest.pkl              R²=0.4191  scaling=no
  baseline/xgboost.pkl                    R²=0.4398  scaling=no

Best model: XGBoost


## 7. Improved Models — Fixing Systematic Underprediction

Three problems with the baseline models:

| Problem | Fix |
|---|---|
| Log-space regression underestimates mean by ~2-3x (Jensen's inequality) | **Duan's Smearing Correction** — multiply predictions by E[exp(residuals)] |
| Zero-inflated target (15% have $0 spending) biases model toward zero | **Two-Stage Hurdle Model** — separate classifier + regressor |
| Log1p transform not ideal for zero-spike + heavy right tail | **XGBoost Tweedie** — native zero-inflated distribution objective |


In [11]:
# ── Duan's Smearing Correction ────────────────────────────────────────────────
# When a model is trained on log1p(y) and predictions are exponentiated,
# Jensen's inequality causes systematic underprediction.
# Fix: corrected_dollar = expm1(log_pred) * E[exp(residuals_train)]
# where residuals are computed on the TRAINING set.

smearing = {}
model_inputs = {
    'Linear Regression': (lr, X_train_sc, y_train),
    'SVR':               (svr, X_train_sc, y_train),
    'Random Forest':     (rf, X_train, y_train),
    'XGBoost':           (xgb, X_train, y_train),
}

actual_dollars_test = np.expm1(y_test)
print('Smearing correction (mean predicted vs actual mean = ${:,.0f}):'.format(actual_dollars_test.mean()))
print(f'{"Model":<22} {"Smearing":>10} {"Uncorrected Mean":>18} {"Corrected Mean":>16} {"Actual Mean":>13}')
print('-' * 83)

for name, (model, X_tr, y_tr) in model_inputs.items():
    train_preds = model.predict(X_tr)
    residuals   = y_tr.values - train_preds          # residuals in log1p space
    sf          = float(np.mean(np.exp(residuals)))  # Duan smearing factor
    smearing[name] = sf

    X_te = X_test_sc if name in ['Linear Regression', 'SVR'] else X_test
    preds_log  = model.predict(X_te)
    uncorr_mean = np.expm1(preds_log).mean()
    corr_mean   = (np.expm1(preds_log) * sf).mean()
    print(f'  {name:<20} {sf:>10.4f} ${uncorr_mean:>15,.0f} ${corr_mean:>13,.0f} ${actual_dollars_test.mean():>10,.0f}')

print()
print('Smearing corrects the mean prediction but cannot recover missing predictive power.')


Smearing correction (mean predicted vs actual mean = $7,690):
Model                    Smearing   Uncorrected Mean   Corrected Mean   Actual Mean
-----------------------------------------------------------------------------------
  Linear Regression       11.2387 $          4,101 $       46,095 $     7,690
  SVR                     10.6921 $          3,831 $       40,959 $     7,690
  Random Forest            3.6654 $          2,561 $        9,385 $     7,690
  XGBoost                  5.0067 $          3,089 $       15,467 $     7,690

Smearing corrects the mean prediction but cannot recover missing predictive power.


## 8. XGBoost Tweedie Regression

The **Tweedie distribution** (variance power p ∈ (1,2)) is purpose-built for zero-inflated,
right-skewed positive data — exactly the structure of healthcare spending.

- p = 1 → Poisson (count data)
- **p = 1.5 → Compound Poisson-Gamma** (zero mass + right-skewed continuous; ideal for health costs)
- p = 2 → Gamma (no zeros allowed)

Trained directly on raw dollar amounts — no log1p transform needed.


In [12]:
print('Training XGBoost Tweedie (p=1.5)...')

y_train_dollars = np.expm1(y_train)
y_test_dollars  = np.expm1(y_test)

xgb_tweedie = XGBRegressor(
    objective='reg:tweedie',
    tweedie_variance_power=1.5,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)
xgb_tweedie.fit(X_train, y_train_dollars,
                eval_set=[(X_test, y_test_dollars)],
                verbose=False)

preds_tw_dollars = np.maximum(xgb_tweedie.predict(X_test), 0)
preds_tw_log     = np.log1p(preds_tw_dollars)   # for consistent log-scale metrics

res_tweedie = evaluate('XGBoost Tweedie', y_test, preds_tw_log)

print(f"  R² (log scale)    : {res_tweedie['r2']:.4f}")
print(f"  RMSE ($)          : ${res_tweedie['rmse_dollar']:,.0f}")
print(f"  MAE  ($)          : ${res_tweedie['mae_dollar']:,.0f}")
print(f"  Predicted mean    : ${preds_tw_dollars.mean():,.0f}  (actual ${y_test_dollars.mean():,.0f})")
print(f"  Predicted median  : ${np.median(preds_tw_dollars):,.0f}  (actual ${np.median(y_test_dollars):,.0f})")


Training XGBoost Tweedie (p=1.5)...
  R² (log scale)    : 0.1476
  RMSE ($)          : $17,885
  MAE  ($)          : $6,725
  Predicted mean    : $5,654  (actual $7,690)
  Predicted median  : $3,373  (actual $1,570)


## 9. Two-Stage Hurdle Model

The **hurdle model** is the theoretically correct approach for zero-inflated data:

- **Stage 1**: `XGBClassifier` — will this person have *any* healthcare spending? (binary)
- **Stage 2**: `XGBRegressor` — given spending > $0, how much? (trained only on spenders)
- **Final prediction**: `P(spend > 0) × smearing-corrected amount`

By training Stage 2 only on the 15,284 spenders (85%), the regressor learns from people
who actually used healthcare — not biased by the zero mass.


In [13]:
from sklearn.linear_model import LogisticRegression

print('Training Two-Stage Hurdle Model...')

# ── Stage 1: zero vs nonzero ───────────────────────────────────────────────
y_binary_train = (y_train > 0).astype(int)
y_binary_test  = (y_test  > 0).astype(int)

clf_hurdle = XGBRegressor.__class__  # import already present
from xgboost import XGBClassifier

clf_hurdle = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
    eval_metric='logloss',
)
clf_hurdle.fit(X_train, y_binary_train,
               eval_set=[(X_test, y_binary_test)],
               verbose=False)

p_spend_test  = clf_hurdle.predict_proba(X_test)[:, 1]
stage1_acc = (clf_hurdle.predict(X_test) == y_binary_test).mean()
print(f'Stage 1 accuracy: {stage1_acc*100:.1f}%  (baseline if predict all-spending: {y_binary_test.mean()*100:.1f}%)')

# ── Stage 2: log1p regression on spenders only ────────────────────────────
spender_mask_train = y_train > 0
print(f'Stage 2 training on {spender_mask_train.sum():,} spenders ({spender_mask_train.mean()*100:.1f}% of train)')

xgb_hurdle_reg = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)
xgb_hurdle_reg.fit(
    X_train[spender_mask_train],
    y_train[spender_mask_train],
    eval_set=[(X_test[y_test > 0], y_test[y_test > 0])],
    verbose=False,
)

# ── Smearing correction for Stage 2 ───────────────────────────────────────
hurdle_train_preds = xgb_hurdle_reg.predict(X_train[spender_mask_train])
hurdle_residuals   = y_train[spender_mask_train].values - hurdle_train_preds
hurdle_smearing    = float(np.mean(np.exp(hurdle_residuals)))
print(f'Stage 2 smearing factor: {hurdle_smearing:.4f}')

# ── Final prediction: P(spend>0) * smearing-corrected amount ──────────────
log_amount_test  = xgb_hurdle_reg.predict(X_test)
dollar_amount    = np.expm1(log_amount_test) * hurdle_smearing
preds_hurdle     = np.maximum(p_spend_test * dollar_amount, 0)
preds_hurdle_log = np.log1p(preds_hurdle)

res_hurdle = evaluate('Two-Stage Hurdle', y_test, preds_hurdle_log)
y_test_dollars = np.expm1(y_test)

print(f"\nTwo-Stage Hurdle:")
print(f"  R² (log scale)    : {res_hurdle['r2']:.4f}")
print(f"  RMSE ($)          : ${res_hurdle['rmse_dollar']:,.0f}")
print(f"  MAE  ($)          : ${res_hurdle['mae_dollar']:,.0f}")
print(f"  Predicted mean    : ${preds_hurdle.mean():,.0f}  (actual ${y_test_dollars.mean():,.0f})")
print(f"  Predicted median  : ${np.median(preds_hurdle):,.0f}  (actual ${np.median(y_test_dollars):,.0f})")


Training Two-Stage Hurdle Model...
Stage 1 accuracy: 88.2%  (baseline if predict all-spending: 85.1%)
Stage 2 training on 15,222 spenders (84.8% of train)
Stage 2 smearing factor: 1.8025

Two-Stage Hurdle:
  R² (log scale)    : 0.1877
  RMSE ($)          : $18,095
  MAE  ($)          : $6,957
  Predicted mean    : $6,231  (actual $7,690)
  Predicted median  : $2,921  (actual $1,570)


In [14]:
y_test_dollars = np.expm1(y_test)

all_results = []

# ── Original models + smearing correction ────────────────────────────────────
for name, (model, _, _) in model_inputs.items():
    sf   = smearing[name]
    X_te = X_test_sc if name in ['Linear Regression', 'SVR'] else X_test
    preds_log    = model.predict(X_te)
    preds_dollar = np.maximum(np.expm1(preds_log) * sf, 0)
    all_results.append({
        'Model': name + ' + Smearing',
        'R² (log)': round(r2_score(y_test, preds_log), 4),
        'RMSE ($)': int(np.sqrt(mean_squared_error(y_test_dollars, preds_dollar))),
        'MAE ($)':  int(mean_absolute_error(y_test_dollars, preds_dollar)),
        'Pred Mean': int(preds_dollar.mean()),
    })

# ── Tweedie ───────────────────────────────────────────────────────────────────
all_results.append({
    'Model':      'XGBoost Tweedie',
    'R² (log)':   round(r2_score(y_test, np.log1p(preds_tw_dollars)), 4),
    'RMSE ($)':   int(np.sqrt(mean_squared_error(y_test_dollars, preds_tw_dollars))),
    'MAE ($)':    int(mean_absolute_error(y_test_dollars, preds_tw_dollars)),
    'Pred Mean':  int(preds_tw_dollars.mean()),
})

# ── Hurdle ────────────────────────────────────────────────────────────────────
all_results.append({
    'Model':      'Two-Stage Hurdle',
    'R² (log)':   round(r2_score(y_test, preds_hurdle_log), 4),
    'RMSE ($)':   int(np.sqrt(mean_squared_error(y_test_dollars, preds_hurdle))),
    'MAE ($)':    int(mean_absolute_error(y_test_dollars, preds_hurdle)),
    'Pred Mean':  int(preds_hurdle.mean()),
})

comp_df = pd.DataFrame(all_results).set_index('Model')
print('=' * 78)
print(f'FINAL MODEL COMPARISON  (actual mean = ${y_test_dollars.mean():,.0f}  median = ${np.median(y_test_dollars):,.0f})'.center(78))
print('=' * 78)
print(comp_df.to_string())
print('=' * 78)

best_row   = comp_df['R² (log)'].idxmax()
best_final = best_row.replace(' + Smearing', '')
print(f'\nBest model by R²: {best_row}')
print(f'Best base model name (for web app): {best_final}')


       FINAL MODEL COMPARISON  (actual mean = $7,690  median = $1,570)        
                              R² (log)  RMSE ($)  MAE ($)  Pred Mean
Model                                                               
Linear Regression + Smearing    0.4049    148835    42049      46095
SVR + Smearing                  0.3794     83895    36484      40959
Random Forest + Smearing        0.4191     19377     8742       9385
XGBoost + Smearing              0.4398     27977    13215      15466
XGBoost Tweedie                 0.1476     17884     6724       5654
Two-Stage Hurdle                0.1877     18095     6956       6230

Best model by R²: XGBoost + Smearing
Best base model name (for web app): XGBoost


In [15]:
import os, json

os.makedirs('../models', exist_ok=True)

# ── Save new models ───────────────────────────────────────────────────────────
joblib.dump(xgb_tweedie,     '../models/hurdle/xgboost_tweedie.pkl')
joblib.dump(clf_hurdle,      '../models/hurdle/hurdle_classifier.pkl')
joblib.dump(xgb_hurdle_reg,  '../models/hurdle/hurdle_regressor.pkl')

# ── Save smearing factors ─────────────────────────────────────────────────────
smearing_info = {
    'smearing_factors': smearing,
    'hurdle_smearing':  hurdle_smearing,
    'note': 'Multiply expm1(log_pred) by smearing_factor to correct log-space underestimation bias (Duan 1983).',
}
with open('../models/meta/smearing_factors.json', 'w') as f:
    json.dump(smearing_info, f, indent=2)

# ── Save MEPS percentile lookup (correct web-app comparison benchmark) ────────
y_all_dollars = np.expm1(pd.concat([y_train, y_test]))
pct_lookup = {str(p): float(np.percentile(y_all_dollars, p)) for p in range(0, 101)}

pct_meta = {
    'meps_mean':    float(y_all_dollars.mean()),
    'meps_median':  float(y_all_dollars.median()),
    'meps_p25':     float(np.percentile(y_all_dollars, 25)),
    'meps_p75':     float(np.percentile(y_all_dollars, 75)),
    'meps_p90':     float(np.percentile(y_all_dollars, 90)),
    'meps_zero_pct': float((y_all_dollars == 0).mean() * 100),
    'percentiles':  pct_lookup,
    'note': 'Use MEPS-based figures for web app comparisons — NOT CMS national per-capita ($13,493) which includes administrative/overhead costs.',
}
with open('../models/meta/percentile_lookup.json', 'w') as f:
    json.dump(pct_meta, f, indent=2)

# ── Update meta/model_meta.json with new models + smearing ────────────────────────
with open('../models/meta/model_meta.json') as f:
    meta = json.load(f)

for name in ['Linear Regression', 'SVR', 'Random Forest', 'XGBoost']:
    if name in meta['models']:
        meta['models'][name]['smearing_factor'] = smearing[name]

meta['models']['XGBoost Tweedie'] = {
    'file': 'hurdle/xgboost_tweedie.pkl',
    'needs_scaling': False,
    'target_space': 'dollar',
    'smearing_factor': None,
}
meta['models']['Two-Stage Hurdle'] = {
    'file_classifier': 'hurdle/hurdle_classifier.pkl',
    'file_regressor':  'hurdle/hurdle_regressor.pkl',
    'needs_scaling':   False,
    'target_space':    'log1p',
    'smearing_factor': hurdle_smearing,
}
meta['best_model'] = best_final
meta['meps_stats'] = {
    'mean':   float(y_all_dollars.mean()),
    'median': float(y_all_dollars.median()),
    'p25':    float(np.percentile(y_all_dollars, 25)),
    'p75':    float(np.percentile(y_all_dollars, 75)),
    'p90':    float(np.percentile(y_all_dollars, 90)),
}

with open('../models/meta/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to models/:')
print('  hurdle/xgboost_tweedie.pkl')
print('  hurdle/hurdle_classifier.pkl')
print('  hurdle/hurdle_regressor.pkl')
print('  meta/smearing_factors.json')
print('  meta/percentile_lookup.json')
print('  meta/model_meta.json  (updated)')
print()
print(f'Best model for web app: {meta["best_model"]}')
print(f'MEPS mean: ${pct_meta["meps_mean"]:,.0f}   median: ${pct_meta["meps_median"]:,.0f}')


Saved to models/:
  hurdle/xgboost_tweedie.pkl
  hurdle/hurdle_classifier.pkl
  hurdle/hurdle_regressor.pkl
  meta/smearing_factors.json
  meta/percentile_lookup.json
  meta/model_meta.json  (updated)

Best model for web app: XGBoost
MEPS mean: $7,639   median: $1,605
